# Info
Of course my first trial of a NEO calculation (FCI ground state energy) wasn't working, so I'm going to try to just do the electronic dof now
Results im trying to reproduce are in https://doi.org/10.1063/5.0150291

# Update
Finished 8/17/26! I am able to get the correct FCI ground state energy for H2 (only the electron part)

In [2]:
import math

from gbasis.integrals.nuclear_electron_attraction import nuclear_electron_attraction_integral
from qiskit_aer import AerSimulator
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper

from qiskit import transpile
from qiskit.circuit import QuantumRegister, QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from gbasis.parsers import parse_nwchem

import numpy as np
import scipy as sp
from gbasis.parsers import parse_gbs, make_contractions
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.overlap_asymm import overlap_integral_asymmetric
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral
from gbasis.integrals.nuclear_electron_attraction import nuclear_electron_attraction_integral

import scipy as sp

So this reproduces the result in Table I for H_el!
-1.15168

In [6]:
import pyscf

dist_a = 0.7414
dist_bohr = dist_a * 1.8897259886

mol = pyscf.M(
    atom = 'H 0 0 0; H 0 0 0.7414',  # in Angstrom
    basis = '6-31g',
    symmetry = True,
)
#=
myuhf = mol.UHF().run()
cisolver = pyscf.fci.FCI(myuhf)
print('E(UHF-FCI) = %.12f' % cisolver.kernel()[0])

converged SCF energy = -1.1267339671166  <S^2> = 2.220446e-16  2S+1 = 1
E(UHF-FCI) = -1.151682732110


In [7]:
myhf = mol.HF().run()

converged SCF energy = -1.12673396711657


In [8]:
elec_basis_dict = parse_nwchem("6-31G.nw")

elec_atoms = ["H", "H"]
elec_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, dist_bohr]])

elec_basis = make_contractions(elec_basis_dict, elec_atoms, elec_atcoords, coord_types="cartesian")

elec_overlap = overlap_integral(elec_basis)
elec_ortho = np.linalg.inv(sp.linalg.sqrtm(elec_overlap))

#elec_ortho = myhf.mo_coeff.T # use MO coeffs from UHF calc instead!

print(overlap_integral(elec_basis, elec_ortho))

elec_ke = kinetic_energy_integral(elec_basis, transform=elec_ortho)

elec_nuclear_attraction = nuclear_electron_attraction_integral(elec_basis, elec_atcoords, np.array([1, 1]), transform=elec_ortho)

elec_elec_coulomb = electron_repulsion_integral(elec_basis, transform=elec_ortho)


[[ 1.00000000e+00 -5.17646051e-16 -1.66651261e-16 -1.85802349e-16]
 [-7.88502938e-16  1.00000000e+00  6.88404133e-16  4.50838428e-17]
 [-8.51940439e-17  7.06066092e-16  1.00000000e+00 -4.61855183e-16]
 [-2.87603970e-16  2.82744763e-16 -4.29272442e-16  1.00000000e+00]]


![alt text](second%20quantized.png "second quantized")
From https://doi.org/10.1063/5.0150291

Terms on the fifth line are irrelevant, they are coulomb interactions invoving the classically treated nuclei.

In [228]:
print(np.round(elec_ortho @ mol.intor('int1e_nuc')  @ elec_ortho.T, 3))
print(np.round(elec_nuclear_attraction, 3))

from pyscf import ao2mo

eri_4fold = ao2mo.kernel(mol, elec_ortho.T)
print(eri_4fold)
print(elec_elec_coulomb[0,0,0,0])

[[-2.531  0.044  0.021 -0.169]
 [ 0.044 -0.849 -0.169 -0.047]
 [ 0.021 -0.169 -2.531  0.044]
 [-0.169 -0.047  0.044 -0.849]]
[[-2.531  0.044  0.021 -0.169]
 [ 0.044 -0.849 -0.169 -0.047]
 [ 0.021 -0.169 -2.531  0.044]
 [-0.169 -0.047  0.044 -0.849]]
[[ 1.19130361e+00  2.72248444e-02  4.66618409e-01  7.94235784e-03
   1.18992329e-02  6.21032128e-01  9.09220875e-02  2.87754709e-02
   1.94175835e-02  3.48458995e-01]
 [ 2.72248444e-02  1.08879372e-02  1.74181170e-02 -1.25952261e-03
  -6.87953808e-04  1.94175835e-02  2.75528306e-03  2.46164094e-03
   1.24818883e-03  9.27535042e-03]
 [ 4.66618409e-01  1.74181170e-02  4.12830335e-01 -2.89405400e-03
  -1.17867422e-02  3.48458995e-01  2.37457633e-02  7.68973937e-03
   9.27535042e-03  2.68867858e-01]
 [ 7.94235784e-03 -1.25952261e-03 -2.89405400e-03  1.47165936e-02
   2.69909044e-03  7.94235784e-03  2.69909044e-03  2.79677113e-03
  -1.25952261e-03 -2.89405400e-03]
 [ 1.18992329e-02 -6.87953808e-04 -1.17867422e-02  2.69909044e-03
   1.27843240e-0

In [229]:
elec_modes = 2*len(elec_basis)

# hamiltonian in terms of annihilation/creation ops. first term we add is a constant, it's the nuclear repulsion (classical coulomb)
h_fermion_op = FermionicOp({'': 1/dist_bohr}, num_spin_orbitals=elec_modes)

# SparsePauliOp identity for  qubits representing the electronic modes
elec_pauli_identity = SparsePauliOp("I"*elec_modes)

mapper = JordanWignerMapper()


for i in range(len(elec_basis)):
    for j in range(len(elec_basis)):
        # Only include terms with the same spin: 2i, 2j and 2i+1, 2j+1 (alpha and beta orbitals respectively)
        h_fermion_op += FermionicOp({f"+_{2*i} -_{2*j}": elec_ke[i, j] + elec_nuclear_attraction[i, j]}, num_spin_orbitals=elec_modes)
        h_fermion_op += FermionicOp({f"+_{2*i+ 1} -_{2*j + 1}": elec_ke[i, j] + elec_nuclear_attraction[i, j]}, num_spin_orbitals=elec_modes)

for i in range(len(elec_basis)):
    for j in range(len(elec_basis)):
        for k in range(len(elec_basis)):
            for l in range(len(elec_basis)):
                for spin1 in range(2):
                    for spin2 in range(2):
                        h_fermion_op += FermionicOp({
                            f"+_{2*i + spin1} +_{2*j + spin2} -_{2*k + spin2} -_{2*l + spin1}": 0.5 * elec_elec_coulomb[i, j, l, k],
                        }, num_spin_orbitals=elec_modes)


h_pauli_op = mapper.map(h_fermion_op)
h_mtx = h_pauli_op.to_matrix()

#h_mtx = np.round(h_mtx, 4)

In [230]:
import itertools

num_electrons = 2

valid_states = [] #binary strings in occ number basis corresponding to states with 2 electorns and 2 protons
valid_states_indices = [] #decimal version of these binary strings

for elec_indices in itertools.combinations(range(elec_modes), num_electrons):
    elec_string = ['0'] * elec_modes

    for i in elec_indices:
        elec_string[i] = '1'

    total_string = "".join(elec_string)

    valid_states.append(total_string)
    valid_states_indices.append(int(total_string, 2))

all_states_indices = list(range(2**(elec_modes)))
invalid_states_indices = [index for index in all_states_indices if index not in valid_states_indices]

In [231]:
a = h_mtx[np.ix_(valid_states_indices, invalid_states_indices)]
print(np.nonzero(a))


(array([ 1,  2,  3,  4,  5,  7,  8,  9, 10, 12, 13, 13, 14, 15, 15, 16, 17,
       18, 18, 19, 20, 21, 22, 22, 23, 24, 25, 26]), array([217, 169, 193, 181, 187, 174, 222, 198, 210, 207,  80, 128, 104,
        92, 117,  98, 114,  58, 151, 163,  53, 160,  69, 139,  75, 136,
        63, 148]))


In [232]:
h_mtx_2p = h_mtx[np.ix_(valid_states_indices, valid_states_indices)]

In [234]:
eigvals, eigvecs = sp.linalg.eig(h_mtx_2p)

In [235]:
print(min(eigvals))

(-1.151682727673427+0j)


# Physicist notation integral
![](physnot.png)
this is coulomb[a,b,c,d]